# Modelos, diagnostico y pronosticos

**Laboratorio 1 — Series de Tiempo (CC3084)**

Pipeline en `src/models.py`. Para cada una de las siete series de la Parte 1:

1. Candidatos ARIMA/SARIMA (rejilla acotada + apoyo de `pmdarima.auto_arima`).
2. Diagnostico de residuos de los tres finalistas por AIC (Ljung-Box, ACF, histograma, Q-Q).
3. Modelos alternativos obligatorios: Prophet, Holt-Winters, suavizamiento exponencial, seasonal naive.
4. Pronostico sobre el conjunto de prueba y metricas MAE/RMSE/MAPE (mas AIC/BIC para ARIMA/SARIMA).
5. Seleccion del mejor modelo por serie.

Requiere que `outputs/parte1/` ya exista (ejecutar antes `notebooks/01_eda_y_series.ipynb`).

Salidas en `outputs/parte2/` (`metricas_modelos.csv`, `mejores_modelos.csv`, `candidatos_arima.csv`, `pronosticos/`, `residuos/`, `figuras/`, `tablas_latex/`, `manifest.md`).

## 1. Configuracion

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import models as M

print("ROOT:", ROOT)
print("PARTE1_DIR:", M.PARTE1_DIR)
print("PARTE2_DIR:", M.PARTE2_DIR)
print("HAS_PMDARIMA:", M.HAS_PMDARIMA)
print("HAS_PROPHET:", M.HAS_PROPHET)

config = M.cargar_configuracion_series()
config[["serie", "d_sugerido", "D_sugerido", "transformacion"]]

## 2. Ejecutar pipeline completo

Ajusta ARIMA/SARIMA, Prophet, Holt-Winters, suavizamiento exponencial y seasonal naive para las siete series. Puede tardar varios minutos.

In [ ]:
resumen = M.ejecutar_parte2()
resumen["mejores"][["serie", "modelo_seleccionado", "MAE", "RMSE", "AIC", "BIC"]]

## 3. Notas y problemas de convergencia

In [ ]:
for serie, notas in resumen["notas_por_serie"].items():
    if notas:
        print(f"--- {serie} ---")
        for n in notas:
            print(" -", n)

## 4. Verificar salidas

In [ ]:
import pandas as pd

metricas = pd.read_csv(M.PARTE2_DIR / "metricas_modelos.csv")
print("Filas de metricas:", len(metricas))
display(metricas.sort_values(["serie", "RMSE"]).head(20))

print("\nPronosticos generados:")
for p in sorted(M.PRONOSTICOS_DIR.glob("*.csv"))[:10]:
    print(" -", p.name)

print("\n--- manifest.md ---\n")
print((M.PARTE2_DIR / "manifest.md").read_text(encoding="utf-8"))